## First small models

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy.spatial import cKDTree

### Load data

In [ ]:
path_flo = "C:/Users/flori/Desktop/pa-1_data/zurich_1.las"

las = laspy.read(path_flo)

# List all available attributes in the LAS file
print("Available attributes in LAS file:")
for spec in las.point_format.dimension_names:
    print("-", spec)

    
df = pd.DataFrame({
    "X": las.x,
    "Y": las.y,
    "Z": las.z,
    "Intensity": las.intensity,
    "Classification": las.classification
})

# Show the first few rows
print(df.head())

# total number of points
print("\nTotal number of points:\n", len(df))

Available attributes in LAS file:
- X
- Y
- Z
- intensity
- return_number
- number_of_returns
- scan_direction_flag
- edge_of_flight_line
- classification
- synthetic
- key_point
- withheld
- scan_angle_rank
- user_data
- point_source_id
- gps_time
                                                   X  \
0  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
1  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
2  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
3  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
4  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   

                                                   Y  \
0  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
1  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
2  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
3  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
4  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   

                                                   Z  Intensity  \
0  [433.53

### Basic PyTorch Model

In [2]:
# Compute ground height (approximate lowest Z value)
ground_height = np.min(las.z)

# Extract Features
X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  
    las.number_of_returns,  # New Feature: Helps distinguish vegetation vs. solid objects
    las.z - ground_height,  # Height normalization
    las.scan_angle_rank  # New Feature: Might help for detecting sloped surfaces
)).T.astype(np.float32)  # Convert to float32

# Extract classification labels and create a mapping
unique_labels = np.unique(las.classification)  # Get unique class values
label_map = {label: idx for idx, label in enumerate(unique_labels)}  # Map to 0-based index
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)  # Convert labels

# Take a larger random sample (e.g., 5 million points)
sample_indices = np.random.choice(X.shape[0], size=12000000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

# === Balance Classes (Undersample Majority Classes) ===
min_class_count = min(np.bincount(y))  # Find smallest class size
balanced_X, balanced_y = [], []
for class_label in np.unique(y):
    class_indices = np.where(y == class_label)[0]
    sampled_indices = resample(class_indices, replace=True, n_samples=min_class_count, random_state=42)
    balanced_X.append(X[sampled_indices])
    balanced_y.append(y[sampled_indices])

X = np.vstack(balanced_X)
y = np.hstack(balanced_y)

# === Normalize Features ===
scaler = StandardScaler()
X = scaler.fit_transform(X)

# === Convert to PyTorch tensors ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# === Define Improved Neural Network ===
class PointCloudNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PointCloudNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)  # Dropout to prevent overfitting

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)  # Dropout applied before final layer
        x = self.fc4(x)
        return x

# Initialize model
input_size = X_train.shape[1]  # Now using all features
num_classes = len(unique_labels)  # Number of unique classifications
model = PointCloudNet(input_size, num_classes)

# === Define Loss, Optimizer, and Learning Rate Scheduler ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # AdamW instead of Adam
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR every 10 epochs

# === Train the Model (50 Epochs) ===
num_epochs = 100
batch_size = 1024

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()  # Adjust learning rate
    
    if (epoch + 1) % 5 == 0:  # Print loss every 5 epochs
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# === Evaluate Model ===
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, axis=1)
    accuracy = (predicted == y_test).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch [5/100], Loss: 1.2580
Epoch [10/100], Loss: 1.0690
Epoch [15/100], Loss: 0.9992
Epoch [20/100], Loss: 0.9498
Epoch [25/100], Loss: 0.9191
Epoch [30/100], Loss: 0.8954
Epoch [35/100], Loss: 0.8805
Epoch [40/100], Loss: 0.8648
Epoch [45/100], Loss: 0.8578
Epoch [50/100], Loss: 0.8495
Epoch [55/100], Loss: 0.8514
Epoch [60/100], Loss: 0.8439
Epoch [65/100], Loss: 0.8432
Epoch [70/100], Loss: 0.8397
Epoch [75/100], Loss: 0.8394
Epoch [80/100], Loss: 0.8382
Epoch [85/100], Loss: 0.8374
Epoch [90/100], Loss: 0.8385
Epoch [95/100], Loss: 0.8385
Epoch [100/100], Loss: 0.8388
Test Accuracy: 0.6923


##### updated version

In [5]:
# === Compute ground height (approximate lowest Z value) ===
ground_height = np.min(las.z)

# === Extract Features (Adding Local Height Variance) ===
# Build a KDTree for nearest-neighbor search
xyz = np.vstack((las.x, las.y, las.z)).T
tree = cKDTree(xyz)

# Compute local height variance for each point
_, idxs = tree.query(xyz, k=10)  # Find 10 nearest neighbors
local_height_var = np.var(las.z[idxs], axis=1)

X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  
    las.number_of_returns, 
    las.z - ground_height,  # Height normalization
    las.scan_angle_rank,
    local_height_var  # New feature: local height variance
)).T.astype(np.float32)

# === Extract classification labels and create a mapping ===
unique_labels = np.unique(las.classification)
label_map = {label: idx for idx, label in enumerate(unique_labels)}
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)

# === Take a larger random sample (e.g., 12 million points) ===
sample_indices = np.random.choice(X.shape[0], size=1200000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

# === Normalize Features ===
scaler = StandardScaler()
X = scaler.fit_transform(X)

# === Convert to PyTorch tensors and split data ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# === Compute Class Weights for Imbalanced Data ===
class_counts = np.bincount(y_train.numpy())  # Get count of each class
class_weights = 1.0 / class_counts  # Inverse frequency
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# === Define Improved Neural Network with Residual Connections ===
class PointCloudNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PointCloudNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        
        self.fc2 = nn.Linear(128, 256)
        self.bn2 = nn.BatchNorm1d(256)
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        
        self.fc4 = nn.Linear(128, num_classes)
        
        self.leaky_relu = nn.LeakyReLU(0.1)  # LeakyReLU instead of ReLU
        self.dropout = nn.Dropout(p=0.2)  # Lowered dropout for better learning

    def forward(self, x):
        x1 = self.leaky_relu(self.bn1(self.fc1(x)))
        x2 = self.leaky_relu(self.bn2(self.fc2(x1)))
        x3 = self.leaky_relu(self.bn3(self.fc3(x2))) + x1  # Residual connection
        x3 = self.dropout(x3)  # Dropout applied before final layer
        return self.fc4(x3)

# Initialize model
input_size = X_train.shape[1]
num_classes = len(unique_labels)
model = PointCloudNet(input_size, num_classes)

# === Define Loss, Optimizer, and Learning Rate Scheduler ===
criterion = nn.CrossEntropyLoss(weight=class_weights)  # Weighted loss
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Dynamic learning rate adjustment
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# === Train the Model (100 Epochs) ===
num_epochs = 100
batch_size = 512  # Reduced for better generalization

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    scheduler.step(loss)  # Reduce LR on plateau
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# === Evaluate Model ===
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, axis=1)
    accuracy = (predicted == y_test).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch [5/100], Loss: 1.2940
Epoch [10/100], Loss: 1.0965
Epoch [15/100], Loss: 0.9917
Epoch [20/100], Loss: 0.9195
Epoch [25/100], Loss: 0.8614
Epoch [30/100], Loss: 0.8092
Epoch [35/100], Loss: 0.7635
Epoch [40/100], Loss: 0.7207
Epoch [45/100], Loss: 0.6810
Epoch [50/100], Loss: 0.6543
Epoch [55/100], Loss: 0.6270
Epoch [60/100], Loss: 0.6054
Epoch [65/100], Loss: 0.5856
Epoch [70/100], Loss: 0.5646
Epoch [75/100], Loss: 0.5504
Epoch [80/100], Loss: 0.5327
Epoch [85/100], Loss: 0.5198
Epoch [90/100], Loss: 0.5021
Epoch [95/100], Loss: 0.4923
Epoch [100/100], Loss: 0.4808
Test Accuracy: 0.8084


### Basic PointNet++ Model

In [5]:
# === Imports and Model Definition ===

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import laspy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


class PointNetSetAbstraction(nn.Module):
    def __init__(self, in_channels, mlp_layers):
        super(PointNetSetAbstraction, self).__init__()
        layers = []
        last_channel = in_channels
        for out_channel in mlp_layers:
            layers.append(nn.Conv1d(last_channel, out_channel, kernel_size=1))
            layers.append(nn.BatchNorm1d(out_channel))
            layers.append(nn.ReLU())
            last_channel = out_channel
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

class PointNetPlusPlus(nn.Module):
    def __init__(self, num_classes):
        super(PointNetPlusPlus, self).__init__()
        self.sa1 = PointNetSetAbstraction(6, [64, 128, 256])
        self.sa2 = PointNetSetAbstraction(256, [256, 512, 1024])
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(p=0.3)
        
    def forward(self, x):
        x = self.sa1(x)
        x = self.sa2(x)
        x = torch.max(x, 2)[0]  # Global max pooling
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x




In [14]:
# === Load and Preprocess LAS File ===

las = laspy.read(path_flo)

ground_height = np.min(las.z)

X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  
    las.z - ground_height  
)).T.astype(np.float32)  

unique_labels = np.unique(las.classification)
label_map = {label: idx for idx, label in enumerate(unique_labels)}
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)

sample_indices = np.random.choice(X.shape[0], size=50000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)  # Keep original shape
X_test = torch.tensor(X_test, dtype=torch.float32)  
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [15]:
# === Train PointNet++ Model ===

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointNetPlusPlus(num_classes=len(unique_labels)).to(device)
X_train, X_test, y_train, y_test = X_train.to(device), X_test.to(device), y_train.to(device), y_test.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Reshape input to (batch_size, num_features, num_points)
        X_batch = X_batch.unsqueeze(2)  # Add a dummy dimension for num_points

        optimizer.zero_grad()
        outputs = model(X_batch)  # Now correctly shaped
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")

    # Reduce Learning Rate after 20 epochs
    if epoch == 15:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.0001


Epoch 1/30:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 1/30: 100%|██████████| 2500/2500 [02:41<00:00, 15.43it/s]


Epoch [1/30], Loss: 1980.4978


Epoch 2/30: 100%|██████████| 2500/2500 [02:47<00:00, 14.96it/s]


Epoch [2/30], Loss: 1788.0371


...

Epoch [28/30], Loss: 1349.8530


Epoch 29/30: 100%|██████████| 2500/2500 [02:54<00:00, 14.34it/s]


Epoch [29/30], Loss: 1353.4645


Epoch 30/30: 100%|██████████| 2500/2500 [02:47<00:00, 14.89it/s]

Epoch [30/30], Loss: 1363.0135


In [17]:
# === Evaluate PointNet++ Model ===

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Ensure correct shape (batch_size, num_features, num_points)
        X_batch = X_batch.unsqueeze(2)  # Add a dummy dimension for num_points
        
        outputs = model(X_batch)  # Pass correctly shaped input
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test Accuracy: {correct / total:.4f}")


Test Accuracy: 0.7559


#### Evaluation
Model is more accurate but is much too slow for our amount of datapoints

### Basic KPConv Model

##### Data preparation

In [3]:
import torch
import laspy
import numpy as np
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# === Load LAS File ===
las = laspy.read(path_flo)

# Compute ground height
ground_height = np.min(las.z)

# Extract Features
X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  
    las.number_of_returns,  
    las.z - ground_height,  
    las.scan_angle_rank  
)).T.astype(np.float32)

# Convert classification labels to zero-based indexing
unique_labels = np.unique(las.classification)
label_map = {label: idx for idx, label in enumerate(unique_labels)}
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)

# Randomly sample 2 million points for training
sample_size = 2000000
sample_indices = np.random.choice(X.shape[0], size=sample_size, replace=False)
X = X[sample_indices]
y = y[sample_indices]

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch Geometric Data format
train_data = Data(
    x=torch.tensor(X_train, dtype=torch.float32),
    y=torch.tensor(y_train, dtype=torch.long)
)
test_data = Data(
    x=torch.tensor(X_test, dtype=torch.float32),
    y=torch.tensor(y_test, dtype=torch.long)
)


c:\Users\flori\anaconda3\envs\PA-1\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] Die angegebene Prozedur wurde nicht gefunden
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\flori\anaconda3\envs\PA-1\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] Die angegebene Prozedur wurde nicht gefunden
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\flori\anaconda3\envs\PA-1\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] Die angegebene Prozedur wurde nicht gefunden
  warnings.warn(
c:\Users\flori\anaconda3\envs\PA-1\lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred w

##### Define KPConv Model

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops
from torch_geometric.nn import global_max_pool

# === Define the KPConv layer ===

class KPConv(MessagePassing):
    def __init__(self, in_channels, out_channels, kernel_size=16):
        super(KPConv, self).__init__(aggr='max')  # Use max pooling aggregation
        self.kernel_size = kernel_size
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        )

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        return self.mlp(x_j)
    

# === Define the KPConv Model ===

class KPConvModel(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(KPConvModel, self).__init__()
        self.kpconv1 = KPConv(in_channels, 64)
        self.kpconv2 = KPConv(64, 128)
        self.kpconv3 = KPConv(128, 256)

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.relu(self.kpconv1(x, edge_index))
        x = F.relu(self.kpconv2(x, edge_index))
        x = F.relu(self.kpconv3(x, edge_index))

        x = global_max_pool(x, data.batch)  # Global pooling
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



##### Train Model

In [7]:
from torch_geometric.loader import DataLoader
import torch.optim as optim
from tqdm import tqdm
from torch_geometric.nn import knn_graph
import torch_cluster



# Define k (number of neighbors)
k = 16  # Adjust this based on dataset density

# Compute edges using kNN (this creates an edge list)
train_data.edge_index = knn_graph(train_data.x, k=k)
test_data.edge_index = knn_graph(test_data.x, k=k)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KPConvModel(in_channels=X_train.shape[1], num_classes=len(unique_labels)).to(device)

# DataLoader
train_loader = DataLoader([train_data], batch_size=1, shuffle=True)
test_loader = DataLoader([test_data], batch_size=1)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch = batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch)
        loss = criterion(outputs, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")


OSError: [WinError 127] Die angegebene Prozedur wurde nicht gefunden

##### Evaluate Model

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        outputs = model(batch)
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == batch.y).sum().item()
        total += batch.y.size(0)

print(f"Test Accuracy: {correct / total:.4f}")
